# GenAI‑Net — Task Presets Tutorial

This notebook showcases a **standardized task configuration layer** for **GenAI‑Net** (RL4CRN), so users can:
- pick a task with a few knobs (e.g., `kind="logic"` + `logic_fn`), or
- plug in a **custom loss** with the same training loop.

> Tip: you can copy/paste any “Task block” below into your own experiments.

---

## 0) Environment setup

If you're working from the repository root:
```bash
pip install -e .
```


In [1]:
# %% [setup]
import os, sys
import numpy as np
from itertools import product

print("Python:", sys.version.split()[0])


Python: 3.10.12


---

## 1) Imports and utilities

We define a **small task factory** (`make_task`) that builds:
- `time_horizon`
- `u_list`
- `IC` object
- `compute_reward(state)` callable compatible with `Environment.get_reward`

You can keep this code in the notebook, or move it to something like `RL4CRN/utils/tasks.py`.


In [2]:
# %% [imports]
from dataclasses import dataclass
from typing import Callable, Optional, Any, Dict, List, Tuple, Union

from RL4CRN.utils.ic import IC

# Rewards
from RL4CRN.rewards.deterministic import dynamic_tracking_error, oscillation_error
from RL4CRN.rewards.stochastic import dynamic_tracking_error_SSA, robust_tracking_loss_SSA


In [3]:
# %% [task-factory]
@dataclass
class TaskSpec:
    name: str
    time_horizon: np.ndarray
    u_list: List[np.ndarray]
    ic: IC
    compute_reward: Callable[[Any], tuple[float, Dict[str, Any]]]
    render_mode: Optional[dict] = None


def make_time_grid(t_f: float = 100.0, n_t: int = 1000) -> np.ndarray:
    return np.linspace(0.0, t_f, n_t, dtype=np.float32)


def build_u_list(
    kind: str,
    *,
    n_inputs: Optional[int] = None,
    p: Optional[int] = None,
    u_values: Optional[List[float]] = None,
    dose_range: Optional[Tuple[float, float, int]] = None,
    u_spec: Optional[tuple] = None,
) -> List[np.ndarray]:
    """Standardized input scenario builder.

    Parameters
    ----------
    kind:
        Task kind: ``logic``, ``tracking``, ``oscillator``, ``dose_response``, ``ssa_tracking``, ``ssa_robust``.
    n_inputs:
        For logic tasks: number of binary inputs.
    p:
        For non-logic tasks: number of input channels.
    u_values:
        Values used for cartesian product grid (tracking/oscillator tasks).
    dose_range:
        ``(u_min, u_max, n)`` for dose-response.
    u_spec:
        Escape hatch:
        - ``('custom', u_list)``
        - ``('grid', values)``
        - ``('linspace', u_min, u_max, n)``

    Returns
    -------
    list[np.ndarray]
        List of inputs ``u`` (each shape ``(p,)``).
    """
    if u_spec is not None:
        tag, *args = u_spec
        if tag == "custom":
            return args[0]
        if tag == "grid":
            values = args[0]
            dim = p if p is not None else n_inputs
            return [np.array(u, dtype=np.float32) for u in product(values, repeat=dim)]
        if tag == "linspace":
            u_min, u_max, n = args
            return [np.array([u], dtype=np.float32) for u in np.linspace(u_min, u_max, n)]
        raise ValueError(f"Unknown u_spec: {u_spec}")

    if kind == "logic":
        if n_inputs is None:
            raise ValueError("logic task needs n_inputs")
        return [np.array(u, dtype=np.float32) for u in product([0.0, 1.0], repeat=n_inputs)]

    if kind in ("tracking", "oscillator", "ssa_tracking", "ssa_robust"):
        if p is None:
            raise ValueError(f"{kind} task needs p")
        values = u_values if u_values is not None else [1.0]
        return [np.array(u, dtype=np.float32) for u in product(values, repeat=p)]

    if kind == "dose_response":
        u_min, u_max, n = dose_range if dose_range is not None else (0.0, 10.0, 10)
        return [np.array([u], dtype=np.float32) for u in np.linspace(u_min, u_max, n)]

    raise ValueError(f"Unknown task kind: {kind}")


def build_ic(species_labels: List[str], ic_spec: Union[str, tuple]) -> IC:
    """Standardized IC builder.

    Parameters
    ----------
    species_labels:
        Names of species in the CRN state.
    ic_spec:
        - ``'zero'``: single IC with all zeros
        - ``('constant', c)``: single IC with all species initialized to ``c``
        - ``('values', values)``: explicit list of IC vectors (list[list[float]])

    Returns
    -------
    IC
        Initial-condition helper used by RL4CRN.
    """
    if ic_spec == "zero":
        return IC(names=species_labels, values=[[0.0 for _ in species_labels]])
    if isinstance(ic_spec, tuple):
        tag = ic_spec[0]
        if tag == "constant":
            val = float(ic_spec[1])
            return IC(names=species_labels, values=[[val for _ in species_labels]])
        if tag == "values":
            return IC(names=species_labels, values=ic_spec[1])
    raise ValueError(f"Unknown ic_spec: {ic_spec}")


def build_weights(q: int, n_t: int, w_spec: Union[str, tuple]) -> np.ndarray:
    """Standardized weights builder for tracking-style losses.

    Parameters
    ----------
    q:
        Output dimension.
    n_t:
        Number of time steps.
    w_spec:
        - ``'steady_state'``: only final time step is weighted
        - ``'uniform'``: all time steps equally weighted
        - ``'transient'``: emphasizes late time, de-emphasizes early time
        - ``('custom', w)``: user-provided weights array

    Returns
    -------
    np.ndarray
        Weight matrix with shape ``(q, n_t)``.
    """
    if w_spec == "steady_state":
        w = np.zeros((q, n_t), dtype=np.float32)
        w[:, -1] = float(n_t)
        return w
    if w_spec == "uniform":
        return np.ones((q, n_t), dtype=np.float32)
    if w_spec == "transient":
        w = np.ones(n_t, dtype=np.float32)
        w[(len(w)//5)*4:] *= 2.0
        w[:(len(w)//5)] *= 0.25
        return w[None, :]
    if isinstance(w_spec, tuple) and w_spec[0] == "custom":
        return np.asarray(w_spec[1], dtype=np.float32)
    raise ValueError(f"Unknown w_spec: {w_spec}")


def make_task(
    kind: str,
    species_labels: List[str],
    *,
    # time
    t_f: float = 100.0,
    n_t: int = 1000,
    # inputs
    n_inputs: Optional[int] = None,
    p: Optional[int] = None,
    u_values: Optional[List[float]] = None,
    dose_range: Optional[Tuple[float, float, int]] = None,
    u_spec: Optional[tuple] = None,
    # IC / weights
    ic: Union[str, tuple] = "zero",
    weights: Union[str, tuple] = "transient",
    # targets
    logic_fn: Optional[Callable[[np.ndarray], bool]] = None,
    target: Union[str, float, None] = None,
    target_fn: Optional[Callable[[float], float]] = None,
    # oscillator knobs
    osc_w: Optional[list] = None,
    t0: float = 20.0,
    # SSA knobs
    n_trajectories: int = 256,
    max_threads: int = 1024,
    cv_weight: float = 1.0,
    rpa_weight: float = 1.0,
) -> TaskSpec:
    """Create a standardized TaskSpec with a minimal set of knobs."""
    time_horizon = make_time_grid(t_f, n_t)
    u_list = build_u_list(kind, n_inputs=n_inputs, p=p, u_values=u_values, dose_range=dose_range, u_spec=u_spec)
    ic_obj = build_ic(species_labels, ic)

    if kind == "logic":
        if logic_fn is None:
            raise ValueError("logic task needs logic_fn")
        r_list = [np.array([float(bool(logic_fn(u)))], dtype=np.float32) for u in u_list]
        w = build_weights(q=1, n_t=n_t, w_spec=weights)

        def compute_reward(state):
            x0_list = ic_obj.get_ic(state)
            return dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

        return TaskSpec("logic", time_horizon, u_list, ic_obj, compute_reward)

    if kind == "tracking":
        if target == "copy_input0":
            r_list = [np.array([u[0]], dtype=np.float32) for u in u_list]
        elif isinstance(target, (int, float)):
            r_list = [np.array([float(target)], dtype=np.float32) for _ in u_list]
        else:
            raise ValueError("tracking needs target='copy_input0' or a constant float target")

        w = build_weights(q=1, n_t=n_t, w_spec=weights)

        def compute_reward(state):
            x0_list = ic_obj.get_ic(state)
            return dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

        return TaskSpec("tracking", time_horizon, u_list, ic_obj, compute_reward)

    if kind == "dose_response":
        if target_fn is None:
            raise ValueError("dose_response needs target_fn(u)->y*")
        w = build_weights(q=1, n_t=n_t, w_spec=weights)

        def compute_reward(state):
            x0_list = ic_obj.get_ic(state)
            r_list = [np.array([target_fn(float(u[0]))], dtype=np.float32) for u in u_list] * len(x0_list)
            return dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

        return TaskSpec("dose_response", time_horizon, u_list, ic_obj, compute_reward)

    if kind == "oscillator":
        mean_list = [np.array([u[0]], dtype=np.float32) for u in u_list]
        w_local = osc_w if osc_w is not None else [0.4, 0.0, 0.2, 0.4]

        def compute_reward(state):
            x0_list = ic_obj.get_ic(state)
            return oscillation_error(state, u_list, x0_list, time_horizon, f_list=None, mean_list=mean_list, w=w_local, t0=t0, LARGE_NUMBER=1e4)

        return TaskSpec("oscillator", time_horizon, u_list, ic_obj, compute_reward)

    if kind == "ssa_tracking":
        if target == "copy_input0":
            r_list = [np.array([u[0]], dtype=np.float32) for u in u_list]
        else:
            raise ValueError("ssa_tracking currently supports target='copy_input0'")
        w = build_weights(q=1, n_t=n_t, w_spec=weights)

        def compute_reward(state):
            x0_list = ic_obj.get_ic(state)
            return dynamic_tracking_error_SSA(
                state, u_list, x0_list, time_horizon, r_list, w,
                n_trajectories=n_trajectories, max_threads=max_threads,
                norm=1, relative=False, LARGE_NUMBER=1e4, LARGE_PENALTY=1e4
            )

        return TaskSpec("ssa_tracking", time_horizon, u_list, ic_obj, compute_reward)

    if kind == "ssa_robust":
        if target == "copy_input0":
            r_list = [np.array([u[0]], dtype=np.float32) for u in u_list]
        else:
            raise ValueError("ssa_robust currently supports target='copy_input0'")
        w = build_weights(q=1, n_t=n_t, w_spec=weights)

        def compute_reward(state):
            x0_list = ic_obj.get_ic(state)
            return robust_tracking_loss_SSA(
                state, u_list, x0_list, time_horizon, r_list, w,
                n_trajectories=n_trajectories, max_threads=max_threads,
                norm=1, relative=True, LARGE_NUMBER=1e3, LARGE_PENALTY=100,
                cv_weight=cv_weight, rpa_weight=rpa_weight
            )

        return TaskSpec("ssa_robust", time_horizon, u_list, ic_obj, compute_reward)

    raise ValueError(f"Unknown kind: {kind}")


---

## 2) Minimal “project scaffold” for examples

In a full run you will:
1. build a **template IOCRN** + reaction library
2. instantiate `Environment` / `ParallelEnvironments`
3. train a policy (REINFORCE/PPO) using `task.compute_reward`

Here we only demonstrate **task construction**.


In [4]:
# %% [example-scaffold]
species_labels_logic = [f"X_{i+1}" for i in range(3)] + ["OUT"]
species_labels_rpa   = ["X_1", "X_2", "X_3", "X_4", "X_5", "X_6", "OUT"]
species_labels_osc   = ["X_1", "X_2", "X_3"]
species_labels_dose  = ["X_1", "X_2", "OUT"]

print("Logic labels:", species_labels_logic)
print("RPA labels:", species_labels_rpa)
print("Osc labels:", species_labels_osc)
print("Dose labels:", species_labels_dose)


Logic labels: ['X_1', 'X_2', 'X_3', 'OUT']
RPA labels: ['X_1', 'X_2', 'X_3', 'X_4', 'X_5', 'X_6', 'OUT']
Osc labels: ['X_1', 'X_2', 'X_3']
Dose labels: ['X_1', 'X_2', 'OUT']


---

## 3) Logic circuits

User knobs:
- `n_inputs`
- `logic_fn`
- `ic`, `weights`


In [5]:
# %% [logic-task]
logic_fn = lambda u: (u[0] and (not u[1])) or bool(u[2])

task_logic = make_task(
    kind="logic",
    species_labels=species_labels_logic,
    n_inputs=3,
    logic_fn=logic_fn,
    ic="zero",
    weights="steady_state",
    t_f=100, n_t=1000,
)

print(task_logic.name)
print("num scenarios:", len(task_logic.u_list))
print("first 5 u:", task_logic.u_list[:5])


logic
num scenarios: 8
first 5 u: [array([0., 0., 0.], dtype=float32), array([0., 0., 1.], dtype=float32), array([0., 1., 0.], dtype=float32), array([0., 1., 1.], dtype=float32), array([1., 0., 0.], dtype=float32)]


---

## 4) Deterministic tracking (RPA-style)

User knobs:
- `p`, `u_values`
- `target="copy_input0"` or constant
- `ic`, `weights`


In [6]:
# %% [tracking-task]
task_tracking = make_task(
    kind="tracking",
    species_labels=species_labels_rpa,
    p=3,
    u_values=[0.5, 1.0, 1.5],
    target="copy_input0",
    ic=("constant", 0.01),
    weights="transient",
    t_f=100, n_t=1000,
)

print(task_tracking.name)
print("num scenarios:", len(task_tracking.u_list))
print("first 3 u:", task_tracking.u_list[:3])


tracking
num scenarios: 27
first 3 u: [array([0.5, 0.5, 0.5], dtype=float32), array([0.5, 0.5, 1. ], dtype=float32), array([0.5, 0.5, 1.5], dtype=float32)]


---

## 5) Oscillator discovery (mean-level targeting)

User knobs:
- `p`, `u_values`
- `ic`
- `osc_w`, `t0`


In [7]:
# %% [oscillator-task]
task_osc = make_task(
    kind="oscillator",
    species_labels=species_labels_osc,
    p=1,
    u_values=[1.0],
    ic=("constant", 0.01),
    t_f=100, n_t=1000,
    osc_w=[0.4, 0.0, 0.2, 0.4],
    t0=20.0,
)

print(task_osc.name)
print("num scenarios:", len(task_osc.u_list))
print("u_list:", task_osc.u_list)


oscillator
num scenarios: 1
u_list: [array([1.], dtype=float32)]


---

## 6) Dose–response matching (target is a function)

User knobs:
- `dose_range`
- `target_fn(u)`
- `ic`, `weights`


In [8]:
# %% [dose-response-task]
def hill_function(u: float, kd=5.0, max_production=50.0, n=6.0) -> float:
    return max_production * (u**n) / (kd**n + u**n)

task_dose = make_task(
    kind="dose_response",
    species_labels=species_labels_dose,
    dose_range=(0.0, 10.0, 10),
    target_fn=hill_function,
    ic=("constant", 0.1),
    weights="transient",
    t_f=100, n_t=1000,
)

print(task_dose.name)
print("num scenarios:", len(task_dose.u_list))
print("first 3 doses:", [u[0] for u in task_dose.u_list[:3]])


dose_response
num scenarios: 10
first 3 doses: [np.float32(0.0), np.float32(1.1111112), np.float32(2.2222223)]


---

## 7) Stochastic SSA tracking (mean trajectory error)

User knobs:
- `p`, `u_values`
- `n_trajectories`, `max_threads`


In [9]:
# %% [ssa-tracking-task]
task_ssa = make_task(
    kind="ssa_tracking",
    species_labels=species_labels_rpa,
    p=2,
    u_values=[1.0, 2.0, 3.0],
    target="copy_input0",
    ic="zero",
    weights="steady_state",
    t_f=100, n_t=200,
    n_trajectories=128,
    max_threads=1024,
)

print(task_ssa.name)
print("num scenarios:", len(task_ssa.u_list))


ssa_tracking
num scenarios: 9


---

## 8) Robust SSA (accuracy + CV penalty)

User knobs:
- `p`, `u_values`
- `rpa_weight`, `cv_weight`
- `n_trajectories`, `max_threads`


In [10]:
# %% [ssa-robust-task]
task_ssa_robust = make_task(
    kind="ssa_robust",
    species_labels=species_labels_rpa,
    p=2,
    u_values=[1.0, 2.0, 3.0],
    target="copy_input0",
    ic="zero",
    weights="steady_state",
    t_f=100, n_t=200,
    n_trajectories=128,
    max_threads=1024,
    rpa_weight=3.0,
    cv_weight=1.0,
)

print(task_ssa_robust.name)
print("num scenarios:", len(task_ssa_robust.u_list))


ssa_robust
num scenarios: 9


---

## 9) Custom task: user-defined loss

If presets aren't enough, define your own `compute_reward(state)` and wrap it in a `TaskSpec`.


In [11]:
# %% [custom-task]
def make_custom_task(species_labels, *, t_f=50.0, n_t=200, u_list=None, ic="zero", name="custom", loss_fn=None):
    time_horizon = make_time_grid(t_f, n_t)
    if u_list is None:
        u_list = [np.array([1.0], dtype=np.float32)]
    ic_obj = build_ic(species_labels, ic)

    if loss_fn is None:
        def loss_fn(state, u_list, x0_list, time_horizon):
            return 0.0

    def compute_reward(state):
        x0_list = ic_obj.get_ic(state)
        performance = float(loss_fn(state, u_list, x0_list, time_horizon))
        info = getattr(state, "last_task_info", {})
        if isinstance(info, dict):
            info = dict(info)
        info.update({"reward": performance, "reward type": "custom"})
        state.last_task_info = info
        return performance, info

    return TaskSpec(name, time_horizon, u_list, ic_obj, compute_reward)


task_custom = make_custom_task(
    species_labels=species_labels_logic,
    name="custom_demo",
    loss_fn=lambda state, u_list, x0_list, time_horizon: 123.0,
)

print(task_custom.name)


custom_demo


---

## 10) Using a `TaskSpec` in training

```python
task = make_task(...)
compute_reward = task.compute_reward

# in training loop:
rewards = mult_env.get_reward(compute_reward)
agent.update(rewards, ...)
```
